In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/factory-spares/spare_parts_data.csv


In [2]:
!pip install pyspark

# **Initializing Pyspark**

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

spark = SparkSession.builder.appName('Spare Parts Manager').getOrCreate()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/16 12:51:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# **Loading Data**

In [4]:
# Its mandatory to handle data with relevant schema

check_schema_df = spark.read.csv("/kaggle/input/factory-spares", inferSchema=True, header = True)
check_schema_df.show(10)

+----+---------------------+----------------+-------------+----------+----------+--------+--------------------------------+--------+
|S.No|Material ID in System|       Part Name|Specification|Brand Name|Unit Price|Currency|Position / Location in Warehouse|Quantity|
+----+---------------------+----------------+-------------+----------+----------+--------+--------------------------------+--------+
|   1|              MAT-162|   PLC CPU 1516 |         NULL|   Siemens|  33870.01|     INR|              Machining - Rack B|    NULL|
|   2|              MAT-123|   Hydraulic Oil|       24V DC| SCHNEIDER|  33186.38|    NULL|               Assembly - Rack A|    71.0|
|   3|              MAT-101|   Vision Camera|     Standard|      NULL|       0.0|     EUR|              Machining - Rack B|    -5.0|
|   4|              MAT-186|Proximity Sensor|      230V AC|  Siemens |      NULL|     EUR|                            NULL|    -5.0|
|   5|              MAT-217|   Hydraulic Oil|       24V DC|       abb

In [5]:
check_schema_df.printSchema()

root
 |-- S.No: integer (nullable = true)
 |-- Material ID in System: string (nullable = true)
 |-- Part Name: string (nullable = true)
 |-- Specification: string (nullable = true)
 |-- Brand Name: string (nullable = true)
 |-- Unit Price: double (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Position / Location in Warehouse: string (nullable = true)
 |-- Quantity: double (nullable = true)



In [6]:
# Here the unit price column needs to be float values.

schema = StructType([
    StructField("S.No", IntegerType(), False),
    StructField("Material ID in System", StringType(), True),
    StructField("Part Name", StringType(), True),
    StructField("Specification", StringType(), False),
    StructField("Brand Name", StringType(), True),
    StructField("Unit Price", FloatType(), True),  # Changed to FloatType
    StructField("Currency", StringType(), True),
    StructField("Position / Location in Warehouse", StringType(), True),
    StructField("Quantity", IntegerType(), True)
])

spare_df = spark.read.csv('/kaggle/input/factory-spares', schema=schema, header = True)
spare_df.printSchema()

root
 |-- S.No: integer (nullable = true)
 |-- Material ID in System: string (nullable = true)
 |-- Part Name: string (nullable = true)
 |-- Specification: string (nullable = true)
 |-- Brand Name: string (nullable = true)
 |-- Unit Price: float (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Position / Location in Warehouse: string (nullable = true)
 |-- Quantity: integer (nullable = true)



In [7]:
# Renaming  

from pyspark.sql.functions import col, when, sum as spark_sum, isnan, isnull

spare_df = spare_df \
    .withColumnRenamed("S.No", "s_no") \
    .withColumnRenamed("Material ID in System", "material_id") \
    .withColumnRenamed("Part Name", "part_name") \
    .withColumnRenamed("Brand Name", "brand_name") \
    .withColumnRenamed("Unit Price", "unit_price") \
    .withColumnRenamed("Position / Location in Warehouse", "location")


# **Data Cleaning Tasks**

In [8]:
# Task 1: Check for null or empty values in any column
# For nulls/NaNs: Use isNull() and isnan() for numerics; for strings, check == '' or isNull()
# Function to count nulls/empties per column

from pyspark.sql.types import StringType, NumericType

exprs = []

for field in spare_df.schema.fields:
    col_name = field.name
    
    if isinstance(field.dataType, NumericType):
        exprs.append(
            spark_sum(
                when(col(col_name).isNull() | isnan(col(col_name)), 1).otherwise(0)
            ).alias(col_name)
        )
    elif isinstance(field.dataType, StringType):
        exprs.append(
            spark_sum(
                when(col(col_name).isNull() | (col(col_name) == ""), 1).otherwise(0)
            ).alias(col_name)
        )

spare_df.select(exprs).show()


+----+-----------+---------+-------------+----------+----------+--------+--------+--------+
|s_no|material_id|part_name|Specification|brand_name|unit_price|Currency|location|Quantity|
+----+-----------+---------+-------------+----------+----------+--------+--------+--------+
|   0|          0|        0|         2008|      1539|      3096|    2929|    1667|   12000|
+----+-----------+---------+-------------+----------+----------+--------+--------+--------+



In [9]:
# Task 2: Check for duplicate rows (exact matches across all columns)

exact_duplicates = spare_df.groupBy(*spare_df.columns).count().filter(col("count") > 1) # * is used when passing list of columns as argument
print("Exact Duplicate Rows (if any):")
exact_duplicates.show()
total_exact_dups = exact_duplicates.select(spark_sum((col("count") - 1))).collect()[0][0] or 0
print(f"Total exact duplicate rows: {total_exact_dups}")


Exact Duplicate Rows (if any):


+----+-----------+---------+-------------+----------+----------+--------+--------+--------+-----+
|s_no|material_id|part_name|Specification|brand_name|unit_price|Currency|location|Quantity|count|
+----+-----------+---------+-------------+----------+----------+--------+--------+--------+-----+
+----+-----------+---------+-------------+----------+----------+--------+--------+--------+-----+

Total exact duplicate rows: 0


In [10]:
# Task 3: Check for duplicate combinations of (Material ID in System, Specification, Brand Name)

combo_cols = ["material_id", "specification", "brand_name"]
combo_duplicates = spare_df.groupBy(*combo_cols).count().filter(col("count") > 1)
print("Combination Duplicates (Material ID, Specification, Brand Name):")
combo_duplicates.show()
total_combo_dups = combo_duplicates.select(spark_sum((col("count") - 1))).collect()[0][0] or 0
print(f"Total combination duplicate rows: {total_combo_dups}")

Combination Duplicates (Material ID, Specification, Brand Name):
+-----------+-------------+----------+-----+
|material_id|specification|brand_name|count|
+-----------+-------------+----------+-----+
|    MAT-145|      230V AC|   Siemens|    3|
|    MAT-297|     Standard| Schneider|    2|
|    MAT-198|     Standard|  Siemens |    3|
|    MAT-136|      230V AC|  Siemens |    4|
|    MAT-158|   High Speed|   Siemens|    2|
|    MAT-294|       24V DC|       ABB|    2|
|    MAT-120|         IP67| SCHNEIDER|    2|
|    MAT-277|      230V AC|       abb|    3|
|    MAT-298|   High Speed| SCHNEIDER|    3|
|    MAT-195|       24V DC| SCHNEIDER|    2|
|    MAT-168|       24V DC| SCHNEIDER|    3|
|    MAT-281|       24V DC|       ABB|    4|
|    MAT-119|   High Speed|       abb|    2|
|    MAT-280|       24V DC|       abb|    5|
|    MAT-140|      230V AC|       ABB|    2|
|    MAT-222|     Standard| Schneider|    2|
|    MAT-178|     Standard|       abb|    2|
|    MAT-263|       24V DC|   SIEME

In [11]:
# Task 4: Check if Unit Price or Quantity ever ≤ 0

invalid_prices = spare_df.filter(col("unit_price") <= 0).count()
invalid_quantities = spare_df.filter(col("quantity") <= 0).count()
print(f"Rows with Unit Price ≤ 0: {invalid_prices}")
print(f"Rows with Quantity ≤ 0: {invalid_quantities}")

# If any found, show them
if invalid_prices > 0:
    spare_df.filter(col("Unit Price") <= 0).show()
if invalid_quantities > 0:
    spare_df.filter(col("Quantity") <= 0).show()


Rows with Unit Price ≤ 0: 5924
Rows with Quantity ≤ 0: 0
+----+-----------+-------------------+-------------+----------+----------+--------+--------------------+--------+
|s_no|material_id|          part_name|Specification|brand_name|unit_price|Currency|            location|Quantity|
+----+-----------+-------------------+-------------+----------+----------+--------+--------------------+--------+
|   3|    MAT-101|      Vision Camera|     Standard|      NULL|       0.0|     EUR|  Machining - Rack B|    NULL|
|   6|    MAT-280|      Hydraulic Oil|     Standard| Schneider|       0.0|     INR|     Assembly–Rack A|    NULL|
|   7|    MAT-216|      Vision Camera|   High Speed|   SIEMENS|    -100.0|     EUR|  Machining - Rack B|    NULL|
|   8|    MAT-265|      Vision Camera|         NULL|       abb|       0.0|     USD| Warehouse-1 - Sh...|    NULL|
|   9|    MAT-263|    Servo Motor 2kW|   High Speed|      NULL|       0.0|     INR|  Machining - Rack B|    NULL|
|  10|    MAT-168|      Vision 

# **Standardization of Data Set**  

In [12]:
# Number of distinct brand names

distinct_brands = spare_df.select("brand_name").distinct().count()
print(f"Number of distinct Brand Names: {distinct_brands}")

# Printing their list
spare_df.select("brand_name").distinct().show(truncate=False)

Number of distinct Brand Names: 8
+----------+
|brand_name|
+----------+
| Siemens  |
|SIEMENS   |
|SCHNEIDER |
|Siemens   |
|ABB       |
|abb       |
|Schneider |
|NULL      |
+----------+



In [13]:
# Number of distinct part names

distinct_partnames = spare_df.select("part_name").distinct().count()
print(f"Number of distinct Part Names: {distinct_partnames}")

# Printing their list
spare_df.select("part_name").distinct().show(truncate=False)

Number of distinct Part Names: 10
+-------------------+
|part_name          |
+-------------------+
|VFD 15kW           |
|Proximity Sensor   |
|PLC CPU 1516       |
|Industrial Cable   |
|PLC CPU 1516       |
|Hydraulic Oil      |
|Vision Camera      |
|Robot Teach Pendant|
|Servo Motor 2kW    |
|Bearing 6205       |
+-------------------+



In [14]:
# Number of distinct locations 

distinct_locations = spare_df.select("location").distinct().count()
print(f"Number of distinct locations: {distinct_locations}")

# Printing their list

spare_df.select("location").distinct().show(truncate=False)

Number of distinct locations: 7
+-----------------------+
|location               |
+-----------------------+
|Assembly - Rack A      |
| Warehouse-1 - Shelf 2 |
|Machining - Rack B     |
|Assembly–Rack A        |
|Paint Shop Rack C      |
|Paint Shop — Rack C    |
|NULL                   |
+-----------------------+



# **Detecting Inconsistencies**

In [15]:
from pyspark.sql.functions import col, lower, collect_set, size, trim, count as spark_count

# Task 1: Same Brand Name with different letter cases (e.g., Siemens vs SIEMENS)
# Group by lowercase brand and collect distinct original cases; filter where >1 distinct cases

brand_case_issues = (
    spare_df
    .withColumn("brand_lower", lower(col("brand_name")))
    .groupBy("brand_lower")
    .agg(collect_set("brand_name").alias("distinct_cases"))
    .filter(size(col("distinct_cases")) > 1)
    .select("brand_lower", "distinct_cases")
)

print("Potential case inconsistencies in Brand Name:")
brand_case_issues.show(truncate=False)

issue_count1 = brand_case_issues.count()
print(f"Number of brands with case issues: {issue_count1}")

Potential case inconsistencies in Brand Name:
+-----------+----------------------+
|brand_lower|distinct_cases        |
+-----------+----------------------+
|siemens    |[SIEMENS, Siemens]    |
|abb        |[abb, ABB]            |
|schneider  |[Schneider, SCHNEIDER]|
+-----------+----------------------+

Number of brands with case issues: 3


In [16]:
# Task 2: Extra leading/trailing spaces in Brand Name, Part Name, Location
location_col = "location"

# Check for each column: count rows where original != trimmed
brand_spaces = spare_df.filter(col("brand_name") != trim(col("brand_name"))).count()
part_spaces = spare_df.filter(col("part_name") != trim(col("part_name"))).count()
location_spaces = spare_df.filter(col(location_col) != trim(col(location_col))).count()

print("\nExtra spaces check:")
print(f"Rows with leading/trailing spaces in Brand Name: {brand_spaces}")
print(f"Rows with leading/trailing spaces in Part Name: {part_spaces}")
print(f"Rows with leading/trailing spaces in Location: {location_spaces}")

#Show rows with spaces (example for Brand Name)
if brand_spaces > 0:
    spare_df.filter(col("brand_name") != trim(col("brand_name"))).select("brand_name", trim(col("brand_name")).alias("trimmed")).show(truncate=False)


Extra spaces check:
Rows with leading/trailing spaces in Brand Name: 1462
Rows with leading/trailing spaces in Part Name: 1205
Rows with leading/trailing spaces in Location: 1735
+----------+-------+
|brand_name|trimmed|
+----------+-------+
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
| Siemens  |Siemens|
+----------+-------+
only showing top 20 rows



In [17]:
# Task 3: Multiple formats for location (e.g., contains "-" vs "–" vs no separator)
# Check for different separators: standard hyphen "-", en-dash "–", em-dash "—", or no separator (just space)

df_with_seps = spare_df.withColumn("has_hyphen", col(location_col).contains("-")) \
                 .withColumn("has_en_dash", col(location_col).contains("–")) \
                 .withColumn("has_em_dash", col(location_col).contains("—")) \
                 .withColumn("has_no_sep", col(location_col).rlike(r"([A-Za-z]+\s+[A-Za-z]+)"))  # Detect space-separated without dash

# Aggregate counts of different separator types

sep_issues = df_with_seps.groupBy().agg(
    spark_count(when(col("has_hyphen") & ~col("has_en_dash") & ~col("has_em_dash"), 1)).alias("hyphen_only"),
    spark_count(when(col("has_en_dash") & ~col("has_hyphen") & ~col("has_em_dash"), 1)).alias("en_dash_only"),
    spark_count(when(col("has_em_dash"), 1)).alias("em_dash_any"),
    spark_count(when(col("has_no_sep") & ~col("has_hyphen") & ~col("has_en_dash") & ~col("has_em_dash"), 1)).alias("no_sep_only")
).collect()[0]

print("\nLocation separator format counts:")
print(f"Rows with standard hyphen '-': {sep_issues['hyphen_only']}")
print(f"Rows with en-dash '–': {sep_issues['en_dash_only']}")
print(f"Rows with em-dash '—': {sep_issues['em_dash_any']}")
print(f"Rows with no separator (space only): {sep_issues['no_sep_only']}")

spare_df.filter(col(location_col).contains("–") | col(location_col).contains("—")).select(location_col).distinct().show(truncate=False)


Location separator format counts:
Rows with standard hyphen '-': 5179
Rows with en-dash '–': 1685
Rows with em-dash '—': 1718
Rows with no separator (space only): 1751
+-------------------+
|location           |
+-------------------+
|Assembly–Rack A    |
|Paint Shop — Rack C|
+-------------------+



# **Feature Engineering**

In [18]:
# Compute total stock value using quantity and price

spare_df = spare_df.withColumn("total_stock_value", col("unit_price") * col("quantity"))
spare_df.select('unit_price','quantity','total_stock_value').show()

+----------+--------+-----------------+
|unit_price|quantity|total_stock_value|
+----------+--------+-----------------+
|  33870.01|    NULL|             NULL|
|  33186.38|    NULL|             NULL|
|       0.0|    NULL|             NULL|
|      NULL|    NULL|             NULL|
|      NULL|    NULL|             NULL|
|       0.0|    NULL|             NULL|
|    -100.0|    NULL|             NULL|
|       0.0|    NULL|             NULL|
|       0.0|    NULL|             NULL|
|    -100.0|    NULL|             NULL|
|    -100.0|    NULL|             NULL|
|  76674.18|    NULL|             NULL|
|      NULL|    NULL|             NULL|
|    -100.0|    NULL|             NULL|
|    -100.0|    NULL|             NULL|
|  24651.58|    NULL|             NULL|
|       0.0|    NULL|             NULL|
|      NULL|    NULL|             NULL|
|       0.0|    NULL|             NULL|
|  37561.89|    NULL|             NULL|
+----------+--------+-----------------+
only showing top 20 rows



In [19]:
# 2. warehouse_area: Extract the area before " - "
from pyspark.sql.functions import *
spare_df = spare_df.withColumn("warehouse_area", split(col("location"), " - ")[0])
spare_df.show()

+----+-----------+-------------------+-------------+----------+----------+--------+--------------------+--------+-----------------+-------------------+
|s_no|material_id|          part_name|Specification|brand_name|unit_price|Currency|            location|Quantity|total_stock_value|     warehouse_area|
+----+-----------+-------------------+-------------+----------+----------+--------+--------------------+--------+-----------------+-------------------+
|   1|    MAT-162|      PLC CPU 1516 |         NULL|   Siemens|  33870.01|     INR|  Machining - Rack B|    NULL|             NULL|          Machining|
|   2|    MAT-123|      Hydraulic Oil|       24V DC| SCHNEIDER|  33186.38|    NULL|   Assembly - Rack A|    NULL|             NULL|           Assembly|
|   3|    MAT-101|      Vision Camera|     Standard|      NULL|       0.0|     EUR|  Machining - Rack B|    NULL|             NULL|          Machining|
|   4|    MAT-186|   Proximity Sensor|      230V AC|  Siemens |      NULL|     EUR|     

In [20]:
# 3. part_category: Map based on rules using Part Name
spare_df = spare_df.withColumn("part_category",
    when(col("part_name").contains("Sensor"), "Sensors")
    .when(col("part_name").contains("PLC") | col("part_name").contains("IO") | col("part_name").contains("HMI"), "Automation")
    .when(col("part_name").contains("Cable"), "Electrical")
    .when(col("part_name").contains("Bearing") | col("part_name").contains("Belt"), "Mechanical")
    .when(col("part_name").contains("Oil") | col("part_name").contains("Coolant"), "Consumables")
    .when(col("part_name").contains("Motor") | col("part_name").contains("VFD"), "Drives")
    .otherwise("Others")
)

spare_df.show()

+----+-----------+-------------------+-------------+----------+----------+--------+--------------------+--------+-----------------+-------------------+-------------+
|s_no|material_id|          part_name|Specification|brand_name|unit_price|Currency|            location|Quantity|total_stock_value|     warehouse_area|part_category|
+----+-----------+-------------------+-------------+----------+----------+--------+--------------------+--------+-----------------+-------------------+-------------+
|   1|    MAT-162|      PLC CPU 1516 |         NULL|   Siemens|  33870.01|     INR|  Machining - Rack B|    NULL|             NULL|          Machining|   Automation|
|   2|    MAT-123|      Hydraulic Oil|       24V DC| SCHNEIDER|  33186.38|    NULL|   Assembly - Rack A|    NULL|             NULL|           Assembly|  Consumables|
|   3|    MAT-101|      Vision Camera|     Standard|      NULL|       0.0|     EUR|  Machining - Rack B|    NULL|             NULL|          Machining|       Others|
|   

# **Inventory Analytics**

In [21]:
# Top 5 Most Expensive Parts by Unit Price

top5_exp_unit_price = spare_df.select("part_name","unit_price").orderBy(col("unit_price").desc()).limit(5)
top5_exp_unit_price.show()

+----------------+----------+
|       part_name|unit_price|
+----------------+----------+
|Proximity Sensor| 149961.69|
|   PLC CPU 1516 | 149957.34|
|   Vision Camera| 149717.69|
|        VFD 15kW|  149659.9|
|   Hydraulic Oil| 149624.02|
+----------------+----------+



In [22]:
# Top 5 most expensive parts by total stock value

top5_exp_stock_val = spare_df.select("part_name","total_stock_value").orderBy(col("total_stock_value").desc()).limit(5)
top5_exp_stock_val.show()

+----------------+-----------------+
|       part_name|total_stock_value|
+----------------+-----------------+
|   PLC CPU 1516 |             NULL|
|   Hydraulic Oil|             NULL|
|   Vision Camera|             NULL|
|Proximity Sensor|             NULL|
|   Hydraulic Oil|             NULL|
+----------------+-----------------+



In [23]:
# Total Inventory Value per Warehouse Area

inventory_per_area = spare_df.groupBy("warehouse_area").agg(sum("total_stock_value").alias("total_area_value"))

In [24]:
# Total Quantity per Part Category

quantity_per_category = spare_df.groupBy("part_category").agg(sum("quantity").alias("total_quantity"))

In [25]:
# Average unit price per brand 

avg_price_brand = spare_df.groupBy("brand_name").agg(avg("unit_price").alias("avg_price"))
avg_price_brand.show()

+----------+------------------+
|brand_name|         avg_price|
+----------+------------------+
|      NULL|25622.182794793815|
|  Siemens |22339.304537225315|
|   SIEMENS|24882.044222112287|
| SCHNEIDER|23476.223758176075|
|   Siemens| 27041.64209188183|
|       ABB|25229.550736413486|
|       abb| 24324.98628161182|
| Schneider|28297.686469460437|
+----------+------------------+



In [26]:
# Brand contributing to highest inventory value

spare_df.groupBy("brand_name").agg(sum("total_stock_value").alias("max_value"))\
        .orderBy(col("max_value").desc()).limit(1).show()

+----------+---------+
|brand_name|max_value|
+----------+---------+
|      NULL|     NULL|
+----------+---------+



In [27]:
# WareHouse Area with maximum stock quantity

max_stock_area = spare_df.groupBy("warehouse_area") \
    .agg(sum("quantity").alias("total_quantity")) \
    .orderBy(col("total_quantity").desc()) \
    .limit(1)

max_stock_area.show()


+--------------+--------------+
|warehouse_area|total_quantity|
+--------------+--------------+
|          NULL|          NULL|
+--------------+--------------+



# **Risk And Operational Analysis**

In [28]:
# Parts with atmost 5 quantity

spare_df.select("part_name", "quantity").filter(col("quantity") < 5).show()

+---------+--------+
|part_name|quantity|
+---------+--------+
+---------+--------+



In [29]:
# High Value but low quantity items

spare_df.select("part_name", "unit_price", "quantity").filter((col("unit_price") > 20000) & (col("quantity") < 3)).orderBy(col("unit_price").desc()).show()

+---------+----------+--------+
|part_name|unit_price|quantity|
+---------+----------+--------+
+---------+----------+--------+



In [30]:
# Parts used in only one warehouse area

single_location_parts = spare_df.groupBy("part_name") \
    .agg(countDistinct("location").alias("warehouse_count")) \
    .filter(col("warehouse_count") == 1)

single_location_parts.show()

+---------+---------------+
|part_name|warehouse_count|
+---------+---------------+
+---------+---------------+



In [31]:
# Computing risk score

spare_df = spare_df.withColumn(
    "risk_score",
    col("unit_price") / col("quantity")
)

spare_df.select("part_name", "unit_price", "quantity", round("risk_score",2)).show(truncate=False)

+-------------------+----------+--------+--------------------+
|part_name          |unit_price|quantity|round(risk_score, 2)|
+-------------------+----------+--------+--------------------+
|PLC CPU 1516       |33870.01  |NULL    |NULL                |
|Hydraulic Oil      |33186.38  |NULL    |NULL                |
|Vision Camera      |0.0       |NULL    |NULL                |
|Proximity Sensor   |NULL      |NULL    |NULL                |
|Hydraulic Oil      |NULL      |NULL    |NULL                |
|Hydraulic Oil      |0.0       |NULL    |NULL                |
|Vision Camera      |-100.0    |NULL    |NULL                |
|Vision Camera      |0.0       |NULL    |NULL                |
|Servo Motor 2kW    |0.0       |NULL    |NULL                |
|Vision Camera      |-100.0    |NULL    |NULL                |
|Robot Teach Pendant|-100.0    |NULL    |NULL                |
|Hydraulic Oil      |76674.18  |NULL    |NULL                |
|Servo Motor 2kW    |NULL      |NULL    |NULL          

In [32]:
# Top 5 risky parts 

top5_risky_parts = (
    spare_df
    .orderBy(col("risk_score").desc())
    .select("part_name", "unit_price", "quantity", round("risk_score",2))
    .limit(5)
)

top5_risky_parts.show(truncate=False)

+----------------+----------+--------+--------------------+
|part_name       |unit_price|quantity|round(risk_score, 2)|
+----------------+----------+--------+--------------------+
|PLC CPU 1516    |33870.01  |NULL    |NULL                |
|Hydraulic Oil   |33186.38  |NULL    |NULL                |
|Vision Camera   |0.0       |NULL    |NULL                |
|Proximity Sensor|NULL      |NULL    |NULL                |
|Hydraulic Oil   |NULL      |NULL    |NULL                |
+----------------+----------+--------+--------------------+



# **Marking of Inventory Items**

In [33]:
# Rank parts within each brand by unit price

from pyspark.sql.window import *

windowSpec = Window.partitionBy("brand_name").orderBy(col("unit_price").desc())

rankedParts = spare_df.withColumn("price_rank",dense_rank().over(windowSpec))

rankedParts.show()

+-----+-----------+-------------------+-------------+----------+----------+--------+--------------------+--------+-----------------+-------------------+-------------+----------+----------+
| s_no|material_id|          part_name|Specification|brand_name|unit_price|Currency|            location|Quantity|total_stock_value|     warehouse_area|part_category|risk_score|price_rank|
+-----+-----------+-------------------+-------------+----------+----------+--------+--------------------+--------+-----------------+-------------------+-------------+----------+----------+
| 3899|    MAT-267|       Bearing 6205|     Standard|      NULL|  149145.3|     USD|     Assembly–Rack A|    NULL|             NULL|    Assembly–Rack A|   Mechanical|      NULL|         1|
| 3506|    MAT-155|Robot Teach Pendant|     Standard|      NULL| 148651.89|     USD|  Machining - Rack B|    NULL|             NULL|          Machining|       Others|      NULL|         2|
| 8519|    MAT-263|      Hydraulic Oil|         IP67|  

In [34]:
# Top 2 costly brands

top_2_per_brand = rankedParts.filter(col("price_rank") <= 2)

top_2_per_brand.show()

+-----+-----------+-------------------+-------------+----------+----------+--------+--------------------+--------+-----------------+-------------------+-------------+----------+----------+
| s_no|material_id|          part_name|Specification|brand_name|unit_price|Currency|            location|Quantity|total_stock_value|     warehouse_area|part_category|risk_score|price_rank|
+-----+-----------+-------------------+-------------+----------+----------+--------+--------------------+--------+-----------------+-------------------+-------------+----------+----------+
| 3899|    MAT-267|       Bearing 6205|     Standard|      NULL|  149145.3|     USD|     Assembly–Rack A|    NULL|             NULL|    Assembly–Rack A|   Mechanical|      NULL|         1|
| 3506|    MAT-155|Robot Teach Pendant|     Standard|      NULL| 148651.89|     USD|  Machining - Rack B|    NULL|             NULL|          Machining|       Others|      NULL|         2|
|  165|    MAT-289|           VFD 15kW|         NULL|  

In [35]:
# Ranking warehouse areas by total inventory value

area_values = spare_df.groupBy("warehouse_area").agg(sum("total_stock_value").alias("area_inventory_value"))

globalWindow = Window.orderBy(col("area_inventory_value").desc())

ranked_warehouses = area_values.withColumn("area_rank", rank().over(globalWindow))

ranked_warehouses.show()

26/01/16 12:51:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/16 12:51:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/16 12:51:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------------+--------------------+---------+
|     warehouse_area|area_inventory_value|area_rank|
+-------------------+--------------------+---------+
|               NULL|                NULL|        1|
|    Assembly–Rack A|                NULL|        1|
|          Machining|                NULL|        1|
|        Warehouse-1|                NULL|        1|
|  Paint Shop Rack C|                NULL|        1|
|           Assembly|                NULL|        1|
|Paint Shop — Rack C|                NULL|        1|
+-------------------+--------------------+---------+



26/01/16 12:51:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/16 12:51:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/16 12:51:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [36]:
pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


# **Presentations**

In [37]:
import matplotlib.pyplt as plt

top_parts_pd = (
    spare_df
    .groupBy("part_name")
    .agg(sum("total_stock_value").alias("total_value"))
    .orderBy(col("total_value").desc())
    .limit(10)
    .toPandas()
)

plt.figure()
plt.bar(top_parts_pd["part_name"], top_parts_pd["total_value"])
plt.xticks(rotation=90)
plt.title("Top 10 Parts by Total Stock Value")
plt.xlabel("Part Name")
plt.ylabel("Inventory Value")
plt.show()


ModuleNotFoundError: No module named 'matplotlib.pyplt'